# 期末專案 -- 圖書管理系統

## 學習目標

1. 將整個課程的 OOP 概念整合到一個完整專案
2. 在真實專案中辨認並應用設計模式
3. 為專案撰寫 pytest 自動化測試
4. 理解如何從小型專案延伸到更大的系統

---

## 一、專案概觀

| 課程概念 | 在專案中的體現 |
|:---------|:---------------|
| 類別與封裝 | `Book` 管理書籍狀態 |
| 繼承與多型 | `Student`、`Faculty` 繼承 `User` |
| 特殊方法 | `__str__`、`__eq__`、`__hash__` |
| 例外處理 | 自訂例外取代錯誤回傳值 |
| 單例模式 | `Catalog` 全域唯一目錄 |
| 觀察者模式 | `Librarian` 監聽目錄變動 |
| 門面模式 | `LibrarySystem` 統一對外介面 |

---

## 二、自訂例外

In [ ]:
class LibraryError(Exception):
    """圖書館系統基礎例外"""

class BookNotAvailableError(LibraryError):
    """書籍無法借出"""

class BorrowLimitError(LibraryError):
    """超過借閱上限"""

class NotFoundError(LibraryError):
    """找不到指定資源"""

> 使用例外而非回傳 `(False, "錯誤訊息")`，錯誤不會被忽略，呼叫端用 try/except 處理。

---

## 三、核心類別

### Book

In [ ]:
class Book:
    def __init__(self, book_id, title, author, isbn, category, copies=1):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.isbn = isbn
        self.category = category
        self.copies = copies
        self.available_copies = copies

    def checkout(self):
        if self.available_copies <= 0:
            raise BookNotAvailableError(f"'{self.title}' 無可借複本")
        self.available_copies -= 1

    def return_copy(self):
        if self.available_copies >= self.copies:
            raise LibraryError(f"'{self.title}' 歸還數超過總複本數")
        self.available_copies += 1

    def __str__(self):
        status = "可借閱" if self.available_copies > 0 else "已借出"
        return f"{self.title} by {self.author} ({status}, {self.available_copies}/{self.copies})"

    def __eq__(self, other):
        return isinstance(other, Book) and self.isbn == other.isbn

    def __hash__(self):
        return hash(self.isbn)

### User / Student / Faculty

借閱上限檢查統一在 `User.borrow_book()`，子類別只設定 `max_books`：

In [ ]:
class User:
    max_books = 5

    def __init__(self, user_id, name, email):
        self.user_id = user_id
        self.name = name
        self.email = email
        self.borrowed_books = []

    def borrow_book(self, book):
        if len(self.borrowed_books) >= self.max_books:
            raise BorrowLimitError(f"{self.name} 已達上限 ({self.max_books} 本)")
        book.checkout()
        self.borrowed_books.append(book)

    def return_book(self, book):
        if book not in self.borrowed_books:
            raise NotFoundError(f"{self.name} 並未借閱 '{book.title}'")
        book.return_copy()
        self.borrowed_books.remove(book)

    def __str__(self):
        return f"{self.name} (ID: {self.user_id}, 已借 {len(self.borrowed_books)} 本)"

class Student(User):
    max_books = 3
    def __init__(self, user_id, name, email, grade):
        super().__init__(user_id, name, email)
        self.grade = grade

class Faculty(User):
    max_books = 10
    def __init__(self, user_id, name, email, department):
        super().__init__(user_id, name, email)
        self.department = department

---

## 四、設計模式

### Singleton -- Catalog

In [ ]:
class Catalog:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._books = {}
            cls._instance._observers = []
        return cls._instance

    def add_book(self, book):
        self._books[book.isbn] = book
        self._notify(f"新書加入: {book.title}")

    def get_book(self, isbn):
        if isbn not in self._books:
            raise NotFoundError(f"ISBN {isbn} 不存在")
        return self._books[isbn]

    def search_by_title(self, keyword):
        return [b for b in self._books.values() if keyword.lower() in b.title.lower()]

    @property
    def all_books(self):
        return list(self._books.values())

    def register_observer(self, observer):
        if observer not in self._observers:
            self._observers.append(observer)

    def _notify(self, message):
        for obs in self._observers:
            obs.update(message)

    def reset(self):
        """重設（僅供測試）"""
        self._books.clear()
        self._observers.clear()

### Observer -- Librarian

In [ ]:
class Librarian:
    def __init__(self, staff_id, name):
        self.staff_id = staff_id
        self.name = name
        self.notifications = []

    def update(self, message):
        self.notifications.append(f"[{self.staff_id}] {message}")

    def generate_report(self, catalog):
        total = len(catalog.all_books)
        available = sum(1 for b in catalog.all_books if b.available_copies > 0)
        out = sum(b.copies - b.available_copies for b in catalog.all_books)
        return f"藏書: {total} 種 | 可借: {available} 種 | 已借出: {out} 冊"

### Facade -- LibrarySystem

In [ ]:
class LibrarySystem:
    def __init__(self, name):
        self.name = name
        self.catalog = Catalog()
        self._users = {}
        self._librarians = {}
        self._next_id = 1

    def add_book(self, title, author, isbn, category, copies=1):
        book = Book(f"B{self._next_id:04d}", title, author, isbn, category, copies)
        self._next_id += 1
        self.catalog.add_book(book)
        return book

    def add_student(self, name, email, grade):
        uid = f"U{self._next_id:04d}"; self._next_id += 1
        user = Student(uid, name, email, grade)
        self._users[uid] = user
        return user

    def add_faculty(self, name, email, department):
        uid = f"U{self._next_id:04d}"; self._next_id += 1
        user = Faculty(uid, name, email, department)
        self._users[uid] = user
        return user

    def add_librarian(self, name):
        sid = f"L{len(self._librarians)+1:04d}"
        lib = Librarian(sid, name)
        self._librarians[sid] = lib
        self.catalog.register_observer(lib)
        return lib

    def checkout_book(self, user_id, isbn):
        user = self._users.get(user_id)
        if not user:
            raise NotFoundError(f"使用者 {user_id} 不存在")
        book = self.catalog.get_book(isbn)
        user.borrow_book(book)
        return f"{user.name} 成功借閱 '{book.title}'"

    def return_book(self, user_id, isbn):
        user = self._users.get(user_id)
        if not user:
            raise NotFoundError(f"使用者 {user_id} 不存在")
        book = self.catalog.get_book(isbn)
        user.return_book(book)
        return f"{user.name} 已歸還 '{book.title}'"

---

## 五、自動化測試（pytest）

In [ ]:
import pytest

@pytest.fixture(autouse=True)
def reset_catalog():
    Catalog().reset()
    yield
    Catalog().reset()

@pytest.fixture
def library():
    return LibrarySystem("測試圖書館")

@pytest.fixture
def sample_book(library):
    return library.add_book("Python 入門", "張三", "978-1-001", "程式設計", copies=2)

@pytest.fixture
def student(library):
    return library.add_student("小明", "ming@test.com", "大一")

class TestBook:
    def test_checkout_decreases(self):
        book = Book("B001", "Test", "Auth", "000", "Cat", copies=2)
        book.checkout()
        assert book.available_copies == 1

    def test_no_copies_raises(self):
        book = Book("B001", "Test", "Auth", "000", "Cat", copies=1)
        book.checkout()
        with pytest.raises(BookNotAvailableError):
            book.checkout()

    def test_equality_by_isbn(self):
        a = Book("B001", "A", "Auth", "SAME", "Cat")
        b = Book("B002", "B", "Auth", "SAME", "Cat")
        assert a == b

class TestBorrowing:
    def test_borrow_success(self, library, sample_book, student):
        msg = library.checkout_book(student.user_id, sample_book.isbn)
        assert "成功借閱" in msg

    def test_student_limit(self, library, student):
        for i in range(3):
            library.add_book(f"Book{i}", "A", f"ISBN-{i}", "C")
            library.checkout_book(student.user_id, f"ISBN-{i}")
        library.add_book("Book3", "A", "ISBN-3", "C")
        with pytest.raises(BorrowLimitError):
            library.checkout_book(student.user_id, "ISBN-3")

    def test_return(self, library, sample_book, student):
        library.checkout_book(student.user_id, sample_book.isbn)
        library.return_book(student.user_id, sample_book.isbn)
        assert sample_book.available_copies == 2
        assert len(student.borrowed_books) == 0

    def test_return_not_borrowed_raises(self, library, sample_book, student):
        with pytest.raises(NotFoundError):
            library.return_book(student.user_id, sample_book.isbn)

class TestCatalog:
    def test_singleton(self):
        assert Catalog() is Catalog()

    def test_search(self, library, sample_book):
        assert len(library.catalog.search_by_title("Python")) == 1

    def test_not_found_raises(self, library):
        with pytest.raises(NotFoundError):
            library.catalog.get_book("FAKE")

class TestObserver:
    def test_notification(self, library):
        lib = library.add_librarian("張管理")
        library.add_book("New", "Auth", "ISBN-N", "Cat")
        assert any("新書加入" in n for n in lib.notifications)

class TestIntegration:
    def test_full_workflow(self, library):
        librarian = library.add_librarian("李管理")
        book = library.add_book("OOP", "王五", "978-2", "CS", copies=1)
        stu = library.add_student("小紅", "hong@test.com", "大二")

        library.checkout_book(stu.user_id, book.isbn)
        assert book.available_copies == 0

        stu2 = library.add_student("小華", "hua@test.com", "大三")
        with pytest.raises(BookNotAvailableError):
            library.checkout_book(stu2.user_id, book.isbn)

        library.return_book(stu.user_id, book.isbn)
        assert book.available_copies == 1

---

## 六、系統展示

In [ ]:
library = LibrarySystem("Python 學習圖書館")
librarian = library.add_librarian("張管理")

book1 = library.add_book("Python 程式設計", "張三", "978-1-001", "程式設計", copies=3)
book2 = library.add_book("資料結構", "李四", "978-1-002", "資訊科學", copies=2)

student = library.add_student("小明", "ming@example.com", "大一")
prof = library.add_faculty("王教授", "wang@example.com", "資工系")

print(library.checkout_book(student.user_id, book1.isbn))
print(library.checkout_book(prof.user_id, book2.isbn))
print(f"\n{book1}")
print(f"{student}")
print(library.return_book(student.user_id, book1.isbn))
print(f"\n{librarian.generate_report(library.catalog)}")

try:
    library.checkout_book("FAKE", book1.isbn)
except NotFoundError as e:
    print(f"\n預期錯誤: {e}")

---

## 七、延伸建議

| 方向 | 做法 | 相關概念 |
|:-----|:-----|:---------|
| 資料持久化 | JSON 或 SQLite | 序列化、Context Manager |
| 圖形介面 | Tkinter GUI | 事件驅動、MVC |
| 預約系統 | 借完排隊等候 | 佇列、觀察者模式進階 |
| 逾期罰款 | 記錄日期算罰金 | datetime、策略模式 |
| REST API | Flask / FastAPI | 裝飾器、路由 |

---

## 八、課程回顧

- [ ] 模組與套件：組織程式碼結構
- [ ] 類別基礎：`__init__`、實例/類別/靜態方法
- [ ] 繼承與多型：程式碼重用與彈性擴充
- [ ] 封裝：屬性裝飾器、私有成員
- [ ] 特殊方法：`__str__`、`__eq__`、運算子重載
- [ ] 抽象基底類別：介面契約
- [ ] 裝飾器：函式/類別裝飾器
- [ ] 迭代器與生成器：惰性計算
- [ ] 設計模式：單例、觀察者、工廠、策略、門面
- [ ] 測試：unittest、pytest、TDD、覆蓋率
- [ ] 例外處理：自訂例外、錯誤傳播

> **下一步**：挑一個延伸方向，用 TDD 實作它。Red-Green-Refactor 會陪你走過整個開發生涯。

---

[<< 回到課程總覽](../00_course_outline.md)